In [ ]:
import json
import logging

from openeo.rest.udp import build_process_dict
from utils import udp_params, urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
decimal_year_of_deforestation = udp_params.DECIMAL_YEAR_OF_DEFORESTATION_DATACUBE

spatial_extent = udp_params.SPATIAL_EXTENT

resample_spatial_resolution = udp_params.SPATIAL_RESOLUTION

cropland_probability_threshold = udp_params.CROPLAND_PROBABILITY_THRESHOLD

In [ ]:
parameters = [
    decimal_year_of_deforestation,
    spatial_extent,
    resample_spatial_resolution,
    cropland_probability_threshold,
]

# UDP

# Load Uganda ADM-4 boundaries

geoboundaries URL https://www.geoboundaries.org/api/current/gbHumanitarian/UGA/ADM4/

In [ ]:
ADM_BOUNDARIES_URL = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbHumanitarian/UGA/ADM4/geoBoundaries-UGA-ADM4.geojson"

adm_boundaries = connection.load_url(
    ADM_BOUNDARIES_URL,
    format="GeoJSON",
)

In [ ]:
adm_boundaries.metadata.dimension_names()

In [ ]:
# adm_boundaries = adm_boundaries.filter_bbox(extent=spatial_extent)
# OpenEoApiError: [400] ProcessParameterInvalid: The value passed for parameter 'data' in process 'filter_bbox' is invalid: Expected raster cube but got vector cube.

In [ ]:
vectorcube_filter_bbox_udf = openeo.UDF.from_file(
    "../udf/vectorcube_filter_bbox.py",
    runtime="Python",
    version="3.11",
    # context set here is ignored! 😠
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
adm_boundaries = adm_boundaries.apply_dimension(
    process=vectorcube_filter_bbox_udf,
    dimension="geometry",
    # try passing context here as well 🤷‍♂️
    context={
        "spatial_extent": spatial_extent,
    },
)

# Load decimal year of deforestation

In [ ]:
# initial spatial filter
# also creates client-side DataCube instances

deforestation_year = connection.datacube_from_process(
    "filter_bbox",
    data=decimal_year_of_deforestation,
    extent=spatial_extent,
)

# load cropland probability

In [ ]:
cropland_probability = connection.load_stac(
    url=urls.MEAN_CROPS_STAC,
    spatial_extent=spatial_extent,
    bands=["crops"],
)

In [ ]:
cropland_probability = utils.drop_hidden_dimension(cropland_probability, "t")

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
cropland_probability = cropland_probability.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
# cropland_mask = cropland_probability.band("crops") > cropland_probability_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gt_cropland_probability_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": cropland_probability_threshold,
    },
)

cropland_mask = (
    cropland_probability.filter_bands("crops")
    .apply(gt_cropland_probability_threshold)
    .convert_data_type("bool")
    .band("crops")
)

merge cropland mask into deforestation year datacube

Bands: [year_of_deforestation, forest_baseline, cropland]

In [ ]:
cropland_mask = cropland_mask.add_dimension("bands", label="cropland", type="bands")

In [ ]:
intermediate_datacube = deforestation_year.merge_cubes(cropland_mask)

# Generate raster KIPs

The CDSE openEO backend has basically no support for processing vector data.
Ideally I would generate each KPI, then merge them together into a single output.
However, this is not possible.

https://forum.dataspace.copernicus.eu/t/merge-vector-cubes/5425/

Essentially, `aggregate_spatial` has to be the last process,
because after that the data is a vector cube, and there is no facility to process it any futher. 😡

Here we use a UDF to generate raster KPI layers, 
where each pixel has value = pixel area (units: ha).
Ready for passing to `aggregate_spatial`.

Doing this in a UDF is an optimisation, rather than using many native openEO band math and `merge_cubes` operations that would make the process graph more complex.

In [ ]:
raster_kpis_udf = openeo.UDF.from_file(
    "../udf/raster_kpis.py",
    runtime="Python",
    version="3.11",
    context={
        "years": [2020, 2021, 2022, 2023, 2024],
        "spatial_resolution": resample_spatial_resolution,
    },
)

In [ ]:
kpis_raster = intermediate_datacube.apply_dimension(
    process=raster_kpis_udf,
    dimension="bands",
)

raster stats

In [ ]:
kpis_vector = kpis_raster.aggregate_spatial(
    geometries=adm_boundaries, reducer=openeo.processes.sum
)

# Serialise UDP

In [ ]:
summary = "Forect loss KPIs"
description = (
    "Aggregate forest-stock and deforestation KPIs by Uganda ADM-4 administrative unit, "
    "from a forest baseline mask and a decimal-year deforestation cube. "
    "1. Spatially filter ADM-4 boundaries, and the deforestation cube to the requested extent. "
    "2. Load cropland probability, resample it onto the same grid, and threshold "
    "it into a cropland mask. "
    "3. Build annual deforestation masks (2020-2024) and the subset of those "
    "pixels that overlap cropland. "
    "4. Convert each pixel mask to area in hectares and sum over ADM-4 geometries. "
    "The returned vector cube contains forest-stock baseline, annual forest loss, "
    "and annual forest-loss-to-cropland areas (ha) per administrative unit."
)

udp_spec = build_process_dict(
    kpis_vector,
    process_id="KPIs",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A vector cube, with columns (bands) for each KPI",
        "schema": {"type": "object", "subtype": "vector-cube"},
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)